# Routes Dataset Cleaning
Cleaning pass for `routes_raw.xls` (AtmoSync project) — validates and standardizes the route dataset (`route_id`, `source`, `destination`, `distance_km`, `estimated_travel_time_hours`).

In [ ]:
import pandas as pd

df = pd.read_csv('routes_raw.csv')
df.shape

## 1. Initial inspection

In [ ]:
df.head()

In [ ]:
df.dtypes

In [ ]:
df.isnull().sum()

## 2. Data quality checks

In [ ]:
# Duplicate rows / route_ids
print('Duplicate rows:', df.duplicated().sum())
print('Duplicate route_id:', df['route_id'].duplicated().sum())

In [ ]:
# Self-loop routes (source == destination)
print('Self-loops:', (df['source'] == df['destination']).sum())

In [ ]:
# Whitespace issues in city names
for col in ['source', 'destination']:
    diff = (df[col] != df[col].str.strip()).sum()
    print(f'{col} whitespace issues:', diff)

In [ ]:
# route_id format check (expects RT####)
df['route_id'].str.match(r'^RT\d{4}$').all()

In [ ]:
# Reverse-pair distance consistency (A->B should roughly match B->A)
pair_dist = {}
inconsistent = []
for _, row in df.iterrows():
    key = tuple(sorted([row['source'], row['destination']]))
    d = round(row['distance_km'], 1)
    if key in pair_dist and pair_dist[key] != d:
        inconsistent.append((key, pair_dist[key], d))
    pair_dist[key] = d
print('Inconsistent reverse-pair distances:', len(inconsistent))
inconsistent[:10]

In [ ]:
# Implied speed outlier check (distance / time)
df['speed_kmh'] = df['distance_km'] / df['estimated_travel_time_hours']
df['speed_kmh'].describe()

## 3. Cleaning pass

In [ ]:
clean = df.drop(columns=['speed_kmh']).copy()

# Trim whitespace, standardize casing
clean['route_id'] = clean['route_id'].str.strip()
clean['source'] = clean['source'].str.strip().str.title()
clean['destination'] = clean['destination'].str.strip().str.title()

# Enforce numeric types
clean['distance_km'] = pd.to_numeric(clean['distance_km'], errors='coerce')
clean['estimated_travel_time_hours'] = pd.to_numeric(clean['estimated_travel_time_hours'], errors='coerce')

# Drop exact duplicate rows and self-loop routes (safety net)
clean = clean.drop_duplicates()
clean = clean[clean['source'] != clean['destination']]

# Sort for consistency
clean = clean.sort_values('route_id').reset_index(drop=True)

clean.shape

In [ ]:
clean.head()

## 4. Export cleaned dataset

In [ ]:
clean.to_csv('routes_cleaned.csv', index=False)
print('Saved routes_cleaned.csv:', clean.shape)